# EXP02 - Financial behaviour and affordability

**Status before execution:** predeclared research experiment. This is the single primary notebook and contains all feature, split, training, prediction, verification, and SHA-256 code. It reads only the supplied competition CSVs and installed libraries; it imports no TRUTH or sibling-experiment Python code.

**Hypothesis:** affordability and loan-structure ratios computed from the official application columns can improve ROC-AUC beyond the same CatBoost model trained on the original supplied predictors.

The competition README bans external data and hand-labeling; the course deck recommends deriving features from existing ones. This test excludes the archived 129.5 fee and 0.0132 rate hypotheses, pooled test/validation statistics, target encodings, prediction overrides, and public-leaderboard tuning. See `reports/COMPETITION_MATERIALS_REVIEW.md` and `EXPERIMENT_PROTOCOL.md`.

## 1. Reproducible environment and preflight

The RTX 4050 is required for this run. The GPU check fails closed if GPU 0 is not identified as an RTX 4050. The output directory is immutable; rerunning requires a new `RUN_ID`. A secondary-seed run must also set `WRITE_PRIMARY_SUMMARY = False` to preserve the primary results.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import platform
import subprocess
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

import catboost
import numpy as np
import pandas as pd
import sklearn
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

TRUTH_ROOT = Path(r"D:\gci\truth")
EXPERIMENT_DIR = TRUTH_ROOT / "experiments" / "exp02_financial_behaviour_replication_seed20261105"
NOTEBOOK_PATH = EXPERIMENT_DIR / "EXP02_FINANCIAL_BEHAVIOUR_SEED20261105.ipynb"
DATA_DIR = Path(r"D:\gci\inpuy")
RUN_ID = "financial_ratios_seed20261105"
WRITE_PRIMARY_SUMMARY = True
RUN_DIR = EXPERIMENT_DIR / "runs" / RUN_ID

ID_COLUMN = "SK_ID_CURR"
TARGET_COLUMN = "TARGET"
N_SPLITS = 5
OUTER_SEED = 20261105
INNER_STOP_FRACTION = 0.10
INNER_STOP_SEED_BASE = 20262105
MODEL_SEED_BASE = 20263105
MAX_ITERATIONS = 2500
EARLY_STOPPING_ROUNDS = 150
MODEL_PARAMS = {
    "depth": 7,
    "learning_rate": 0.045,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "l2_leaf_reg": 5.0,
    "random_strength": 0.5,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.8,
    "border_count": 128,
    "task_type": "GPU",
    "devices": "0",
    "thread_count": 8,
    "allow_writing_files": False,
}
EXPECTED_INPUT_SHA256 = {
    "train.csv": "b8dd5a61771d8eb2821972c1b7d3f2f747cc66471f4d34153036d2d9c458ecf1",
    "test.csv": "dfb7aeb96e5af3d8616acaaba6b23cec8680845cfd525f7c08de15c6d2d42c65",
    "sample_submission.csv": "6e5ad587b7c0b3d9fa39c90ce03e161ff5ef292193e117a6a02b11075dffe7db",
}

for required_path in (
    DATA_DIR / "train.csv",
    DATA_DIR / "test.csv",
    DATA_DIR / "sample_submission.csv",
    NOTEBOOK_PATH,
    EXPERIMENT_DIR / "EXPERIMENT_PROTOCOL.md",
):
    if not required_path.is_file():
        raise FileNotFoundError(f"Required standalone experiment input missing: {required_path}")
if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
    raise FileExistsError(f"Run directory is not empty; set a new RUN_ID: {RUN_DIR}")
if WRITE_PRIMARY_SUMMARY:
    for artifact_name in (
        "RESULTS.json",
        "FOLD_METRICS.csv",
        "FEATURE_MANIFEST.json",
        "EXP02_REPORT.md",
        "SHA256_MANIFEST.json",
    ):
        artifact_path = EXPERIMENT_DIR / artifact_name
        if artifact_path.exists():
            raise FileExistsError(
                f"Primary artifacts already exist; preserve them and use a separate run: {artifact_path}"
            )

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def gpu_snapshot() -> list[dict[str, object]]:
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
            "--format=csv,noheader,nounits",
        ],
        check=True,
        capture_output=True,
        text=True,
        timeout=15,
    )
    snapshots = []
    for row in csv.reader(result.stdout.splitlines(), skipinitialspace=True):
        if len(row) != 5:
            raise RuntimeError(f"Unexpected nvidia-smi output row: {row}")
        snapshots.append(
            {
                "index": int(row[0]),
                "name": row[1].strip(),
                "memory_used_mib": int(row[2]),
                "memory_total_mib": int(row[3]),
                "utilization_gpu_pct": int(row[4]),
            }
        )
    if not snapshots:
        raise RuntimeError("nvidia-smi returned no GPUs")
    return snapshots

gpu_before = gpu_snapshot()
gpu0_before = next(
    (item for item in gpu_before if item["index"] == 0), None
)
if gpu0_before is None or "RTX 4050" not in str(gpu0_before["name"]):
    raise RuntimeError(f"Required RTX 4050 on GPU 0 not detected: {gpu_before}")

print("TRUTH root:", TRUTH_ROOT)
print("Data directory:", DATA_DIR)
print("Run directory:", RUN_DIR)
print("GPU before training:", gpu_before)
print("Python:", platform.python_version())
print(
    "NumPy:",
    np.__version__,
    "pandas:",
    pd.__version__,
    "scikit-learn:",
    sklearn.__version__,
    "CatBoost:",
    catboost.__version__,
)


--- code cell 2 start ---
TRUTH root: D:\gci\truth
Data directory: D:\gci\inpuy
Run directory: D:\gci\truth\experiments\exp02_financial_behaviour_replication_seed20261105\runs\financial_ratios_seed20261105
GPU before training: [{'index': 0, 'name': 'NVIDIA GeForce RTX 4050 Laptop GPU', 'memory_used_mib': 95, 'memory_total_mib': 6141, 'utilization_gpu_pct': 0}]
Python: 3.12.10
NumPy: 2.5.4 pandas: 3.1.0 scikit-learn: 1.9.1 CatBoost: 1.2.10


## 2. Load and validate the official inputs

The input hashes must match the preserved course-provided files. IDs are checked for missingness, uniqueness, train/test overlap, and exact sample/test order. `SK_ID_CURR` and `TARGET` are never predictors.

In [2]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sample_path = DATA_DIR / "sample_submission.csv"
input_paths = {
    "train.csv": train_path,
    "test.csv": test_path,
    "sample_submission.csv": sample_path,
}
input_sha256 = {name: sha256_file(path) for name, path in input_paths.items()}
if input_sha256 != EXPECTED_INPUT_SHA256:
    raise ValueError(f"Input files differ from the preserved competition package: {input_sha256}")

train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)
if list(sample_submission.columns) != [ID_COLUMN, TARGET_COLUMN]:
    raise ValueError(f"Unexpected sample submission schema: {sample_submission.columns.tolist()}")
if TARGET_COLUMN not in train_raw or TARGET_COLUMN in test_raw:
    raise ValueError("Training/test target schema is invalid")
if list(train_raw.drop(columns=[TARGET_COLUMN]).columns) != list(test_raw.columns):
    raise ValueError("Train predictors and test columns/order do not match")
for name, frame in (
    ("train", train_raw),
    ("test", test_raw),
    ("sample_submission", sample_submission),
):
    if frame[ID_COLUMN].isna().any():
        raise ValueError(f"Missing IDs in {name}")
    if frame[ID_COLUMN].duplicated().any():
        raise ValueError(f"Duplicate IDs in {name}")
if set(train_raw[ID_COLUMN]).intersection(test_raw[ID_COLUMN]):
    raise ValueError("Train/test IDs overlap")
if not np.array_equal(
    sample_submission[ID_COLUMN].to_numpy(), test_raw[ID_COLUMN].to_numpy()
):
    raise ValueError("Sample submission ID order does not match test data")

labels = train_raw[TARGET_COLUMN].to_numpy(dtype=np.int8)
if not set(np.unique(labels)).issubset({0, 1}):
    raise ValueError("TARGET must be binary 0/1")
if np.isnan(labels).any():
    raise ValueError("TARGET contains missing values")

predictor_columns = list(test_raw.columns)
print(
    f"Train: {train_raw.shape}; test: {test_raw.shape}; "
    f"predictors (including ID): {len(predictor_columns)}"
)
print(
    "Class counts:",
    {int(value): int(count) for value, count in zip(*np.unique(labels, return_counts=True))},
)
print("Input SHA-256:", json.dumps(input_sha256, indent=2))


--- code cell 4 start ---
Train: (171202, 34); test: (61500, 33); predictors (including ID): 33
Class counts: {0: 157381, 1: 13821}
Input SHA-256: {
  "train.csv": "b8dd5a61771d8eb2821972c1b7d3f2f747cc66471f4d34153036d2d9c458ecf1",
  "test.csv": "dfb7aeb96e5af3d8616acaaba6b23cec8680845cfd525f7c08de15c6d2d42c65",
  "sample_submission.csv": "6e5ad587b7c0b3d9fa39c90ce03e161ff5ef292193e117a6a02b11075dffe7db"
}


## 3. Build and unit-check the predeclared financial feature block

Ratios use only fields described in the supplied column dictionary. Invalid or non-positive denominators and non-finite inputs produce `NaN`; no global statistics are fit. The tests check invalid denominators, finite outputs, and invariance to labels.

In [3]:
FINANCIAL_FORMULAS = {
    "FIN_CREDIT_TO_INCOME": "AMT_CREDIT / AMT_INCOME_TOTAL",
    "FIN_ANNUITY_TO_INCOME": "AMT_ANNUITY / AMT_INCOME_TOTAL",
    "FIN_GOODS_TO_CREDIT": "AMT_GOODS_PRICE / AMT_CREDIT",
    "FIN_CREDIT_TO_ANNUITY": "AMT_CREDIT / AMT_ANNUITY",
    "FIN_INCOME_PER_FAMILY_MEMBER": "AMT_INCOME_TOTAL / CNT_FAM_MEMBERS",
    "FIN_CREDIT_PER_FAMILY_MEMBER": "AMT_CREDIT / CNT_FAM_MEMBERS",
    "FIN_ANNUITY_PER_FAMILY_MEMBER": "AMT_ANNUITY / CNT_FAM_MEMBERS",
}
CATEGORICAL_COLUMNS = [
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE",
]
REQUIRED_FINANCIAL_COLUMNS = {
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_INCOME_TOTAL",
    "AMT_GOODS_PRICE",
    "CNT_FAM_MEMBERS",
}
if not REQUIRED_FINANCIAL_COLUMNS.issubset(test_raw.columns):
    raise ValueError("Supplied fields required for the financial feature block are missing")
if not set(CATEGORICAL_COLUMNS).issubset(test_raw.columns):
    raise ValueError("Expected supplied categorical columns are missing")

def safe_ratio(numerator: pd.Series, denominator: pd.Series) -> np.ndarray:
    num = pd.to_numeric(numerator, errors="coerce").to_numpy(dtype=np.float64)
    den = pd.to_numeric(denominator, errors="coerce").to_numpy(dtype=np.float64)
    result = np.full(num.shape, np.nan, dtype=np.float64)
    valid = np.isfinite(num) & np.isfinite(den) & (den > 0.0)
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        result[valid] = num[valid] / den[valid]
    result[~np.isfinite(result)] = np.nan
    return result

def build_financial_features(raw: pd.DataFrame) -> pd.DataFrame:
    if not REQUIRED_FINANCIAL_COLUMNS.issubset(raw.columns):
        missing = sorted(REQUIRED_FINANCIAL_COLUMNS - set(raw.columns))
        raise ValueError(f"Missing financial inputs: {missing}")
    features = pd.DataFrame(
        {
            "FIN_CREDIT_TO_INCOME": safe_ratio(
                raw["AMT_CREDIT"], raw["AMT_INCOME_TOTAL"]
            ),
            "FIN_ANNUITY_TO_INCOME": safe_ratio(
                raw["AMT_ANNUITY"], raw["AMT_INCOME_TOTAL"]
            ),
            "FIN_GOODS_TO_CREDIT": safe_ratio(
                raw["AMT_GOODS_PRICE"], raw["AMT_CREDIT"]
            ),
            "FIN_CREDIT_TO_ANNUITY": safe_ratio(
                raw["AMT_CREDIT"], raw["AMT_ANNUITY"]
            ),
            "FIN_INCOME_PER_FAMILY_MEMBER": safe_ratio(
                raw["AMT_INCOME_TOTAL"], raw["CNT_FAM_MEMBERS"]
            ),
            "FIN_CREDIT_PER_FAMILY_MEMBER": safe_ratio(
                raw["AMT_CREDIT"], raw["CNT_FAM_MEMBERS"]
            ),
            "FIN_ANNUITY_PER_FAMILY_MEMBER": safe_ratio(
                raw["AMT_ANNUITY"], raw["CNT_FAM_MEMBERS"]
            ),
        },
        index=raw.index,
    ).astype(np.float32)
    if np.isinf(features.to_numpy(dtype=np.float64)).any():
        raise ValueError("Financial features contain infinite values")
    return features

def prepare_model_frame(raw: pd.DataFrame, include_financial: bool) -> pd.DataFrame:
    frame = raw.drop(columns=[ID_COLUMN, TARGET_COLUMN], errors="ignore").copy()
    frame.reset_index(drop=True, inplace=True)
    for column in CATEGORICAL_COLUMNS:
        frame[column] = (
            frame[column].astype("string").fillna("__MISSING__").astype(str)
        )
    for column in frame.columns:
        if column not in CATEGORICAL_COLUMNS:
            frame[column] = pd.to_numeric(frame[column], errors="raise").astype(
                np.float32
            )
    frame.replace([np.inf, -np.inf], np.nan, inplace=True)
    if include_financial:
        additions = build_financial_features(raw.reset_index(drop=True))
        frame = pd.concat([frame, additions], axis=1)
    if frame.columns.duplicated().any():
        raise ValueError("Duplicate model feature names")
    if ID_COLUMN in frame or TARGET_COLUMN in frame:
        raise ValueError("ID or target leaked into the model feature frame")
    numeric = frame.select_dtypes(include=[np.number]).to_numpy(dtype=np.float64)
    if np.isinf(numeric).any():
        raise ValueError("Model features contain infinities")
    return frame

toy = pd.DataFrame(
    {
        "AMT_CREDIT": [100.0, 100.0, 100.0],
        "AMT_ANNUITY": [10.0, 0.0, np.nan],
        "AMT_INCOME_TOTAL": [50.0, 0.0, 50.0],
        "AMT_GOODS_PRICE": [80.0, 100.0, 120.0],
        "CNT_FAM_MEMBERS": [2.0, 0.0, np.nan],
        "TARGET": [0, 1, 0],
    }
)
toy_features = build_financial_features(toy)
assert np.isclose(toy_features.loc[0, "FIN_CREDIT_TO_INCOME"], 2.0)
assert np.isclose(toy_features.loc[0, "FIN_CREDIT_TO_ANNUITY"], 10.0)
assert pd.isna(toy_features.loc[1, "FIN_CREDIT_TO_INCOME"])
assert pd.isna(toy_features.loc[1, "FIN_CREDIT_PER_FAMILY_MEMBER"])
assert not np.isinf(toy_features.to_numpy(dtype=np.float64)).any()
pd.testing.assert_frame_equal(
    toy_features,
    build_financial_features(toy.assign(TARGET=[1, 0, 1])),
)

X_raw = prepare_model_frame(train_raw, include_financial=False)
X_test_raw = prepare_model_frame(test_raw, include_financial=False)
X_financial = prepare_model_frame(train_raw, include_financial=True)
X_test_financial = prepare_model_frame(test_raw, include_financial=True)
if list(X_raw.columns) != list(X_test_raw.columns):
    raise ValueError("Raw model train/test feature columns differ")
if list(X_financial.columns) != list(X_test_financial.columns):
    raise ValueError("Financial model train/test feature columns differ")
if list(X_financial.columns[: len(X_raw.columns)]) != list(X_raw.columns):
    raise ValueError("Financial model changed raw feature order")
if list(X_financial.columns[len(X_raw.columns) :]) != list(FINANCIAL_FORMULAS):
    raise ValueError("Unexpected financial feature order")

print("Feature unit tests: PASS")
print("Raw predictors excluding ID:", len(X_raw.columns))
print("Financial features:", list(FINANCIAL_FORMULAS))
print(
    "Added feature missing counts:",
    X_financial[list(FINANCIAL_FORMULAS)].isna().sum().to_dict(),
)


--- code cell 6 start ---
D:\gci\truth\experiments\exp02_financial_behaviour_replication_seed20261105\EXP02_FINANCIAL_BEHAVIOUR_SEED20261105.ipynb#cell-6:78: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
D:\gci\truth\experiments\exp02_financial_behaviour_replication_seed20261105\EXP02_FINANCIAL_BEHAVIOUR_SEED20261105.ipynb#cell-6:78: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
D:\gci\truth\experiments\exp02_financial_behaviour_replication_seed20261105\EXP02_FINANCIAL_BEHAVIOUR_SEED20261105.ipynb#cell-6:78: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See ht

## 4. Freeze outer and inner splits; define GPU training

The baseline and candidate share each outer fold and inner stopping split. Early stopping sees only the inner stopping subset; the final refit sees the complete outer training fold and no validation rows. Model configuration and seeds are fixed before comparison.

In [4]:
fold_ids = np.full(len(labels), -1, dtype=np.int8)
fold_partitions = []
outer_cv = StratifiedKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=OUTER_SEED
)
for fold_number, (outer_fit, outer_valid) in enumerate(
    outer_cv.split(np.zeros(len(labels)), labels), start=1
):
    if np.intersect1d(outer_fit, outer_valid).size:
        raise ValueError(f"Outer fold {fold_number} overlaps")
    fold_ids[outer_valid] = fold_number
    inner_cv = StratifiedShuffleSplit(
        n_splits=1,
        test_size=INNER_STOP_FRACTION,
        random_state=INNER_STOP_SEED_BASE + fold_number,
    )
    inner_fit_rel, inner_stop_rel = next(
        inner_cv.split(np.zeros(len(outer_fit)), labels[outer_fit])
    )
    inner_fit = outer_fit[inner_fit_rel]
    inner_stop = outer_fit[inner_stop_rel]
    if np.intersect1d(inner_fit, inner_stop).size:
        raise ValueError(f"Inner split {fold_number} overlaps")
    if not np.isin(inner_fit, outer_fit).all() or not np.isin(
        inner_stop, outer_fit
    ).all():
        raise ValueError(f"Inner split {fold_number} escaped outer training rows")
    if np.intersect1d(
        outer_valid, np.concatenate([inner_fit, inner_stop])
    ).size:
        raise ValueError(f"Outer validation leaked into inner split {fold_number}")
    fold_partitions.append(
        {
            "fold": fold_number,
            "outer_fit": outer_fit,
            "outer_valid": outer_valid,
            "inner_fit": inner_fit,
            "inner_stop": inner_stop,
        }
    )
fold_counts = np.bincount(fold_ids, minlength=N_SPLITS + 1)[1:]
if (fold_ids < 1).any() or (fold_counts == 0).any():
    raise ValueError("Invalid outer fold assignment coverage")

def fit_one_variant(
    X: pd.DataFrame,
    X_apply: pd.DataFrame,
    y: np.ndarray,
    partition: dict[str, object],
    model_seed: int,
):
    outer_fit = partition["outer_fit"]
    outer_valid = partition["outer_valid"]
    inner_fit = partition["inner_fit"]
    inner_stop = partition["inner_stop"]
    start = time.perf_counter()
    stopping_model = CatBoostClassifier(
        iterations=MAX_ITERATIONS,
        random_seed=model_seed,
        **MODEL_PARAMS,
    )
    stopping_model.fit(
        X.iloc[inner_fit],
        y[inner_fit],
        cat_features=CATEGORICAL_COLUMNS,
        eval_set=(X.iloc[inner_stop], y[inner_stop]),
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        use_best_model=True,
        verbose=False,
    )
    selected_iterations = int(stopping_model.get_best_iteration()) + 1
    if selected_iterations < 1 or selected_iterations > MAX_ITERATIONS:
        raise RuntimeError(
            f"Invalid selected iteration count: {selected_iterations}"
        )
    final_model = CatBoostClassifier(
        iterations=selected_iterations,
        random_seed=model_seed,
        **MODEL_PARAMS,
    )
    final_model.fit(
        X.iloc[outer_fit],
        y[outer_fit],
        cat_features=CATEGORICAL_COLUMNS,
        verbose=False,
    )
    valid_prediction = final_model.predict_proba(X.iloc[outer_valid])[
        :, 1
    ].astype(np.float32)
    apply_prediction = final_model.predict_proba(X_apply)[:, 1].astype(
        np.float32
    )
    return (
        valid_prediction,
        apply_prediction,
        selected_iterations,
        time.perf_counter() - start,
    )

class GpuSampler:
    def __init__(self):
        self.samples = []
        self.errors = []
        self.stop_event = threading.Event()
        self.thread = threading.Thread(
            target=self._sample, name="gpu-telemetry", daemon=True
        )

    def _sample(self):
        while not self.stop_event.is_set():
            try:
                self.samples.extend(gpu_snapshot())
            except Exception as error:
                self.errors.append(f"{type(error).__name__}: {error}")
            self.stop_event.wait(1.0)

    def start(self):
        self.thread.start()

    def stop(self):
        self.stop_event.set()
        self.thread.join(timeout=20)
        if self.thread.is_alive():
            raise RuntimeError("GPU telemetry sampler did not stop")
        if self.errors:
            raise RuntimeError(f"GPU telemetry collection failed: {self.errors}")
        if not self.samples:
            raise RuntimeError("No in-training GPU telemetry samples were captured")

print("Outer folds:", N_SPLITS, "seed:", OUTER_SEED)
print(
    "Inner stopping fraction:",
    INNER_STOP_FRACTION,
    "seed rule: 20261102 + 1-based fold",
)
print(
    "CatBoost GPU parameters:",
    json.dumps(
        {
            **MODEL_PARAMS,
            "max_iterations": MAX_ITERATIONS,
            "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
        },
        indent=2,
    ),
)
print("Fold/inner split isolation checks: PASS")


--- code cell 8 start ---
Outer folds: 5 seed: 20261105
Inner stopping fraction: 0.1 seed rule: 20261102 + 1-based fold
CatBoost GPU parameters: {
  "depth": 7,
  "learning_rate": 0.045,
  "loss_function": "Logloss",
  "eval_metric": "AUC",
  "l2_leaf_reg": 5.0,
  "random_strength": 0.5,
  "bootstrap_type": "Bernoulli",
  "subsample": 0.8,
  "border_count": 128,
  "task_type": "GPU",
  "devices": "0",
  "thread_count": 8,
  "allow_writing_files": false,
  "max_iterations": 2500,
  "early_stopping_rounds": 150
}
Fold/inner split isolation checks: PASS


## 5. Train matched baseline and financial-feature models

This cell performs ten GPU trainings for iteration selection and ten outer-fold refits. It records per-fold AUCs, paired improvements, selected iterations, fit times, and sampled GPU utilization/VRAM. No submission file is generated.

In [5]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
run_started_utc = datetime.now(timezone.utc).isoformat()
run_start = time.perf_counter()
oof_raw = np.full(len(labels), np.nan, dtype=np.float32)
oof_financial = np.full(len(labels), np.nan, dtype=np.float32)
test_raw_by_fold = []
test_financial_by_fold = []
fold_training_records = []
gpu_sampler = GpuSampler()
gpu_sampler.start()
try:
    for partition in fold_partitions:
        fold_number = int(partition["fold"])
        seed = MODEL_SEED_BASE + fold_number
        raw_valid, raw_test_prediction, raw_iterations, raw_seconds = (
            fit_one_variant(
                X_raw, X_test_raw, labels, partition, seed
            )
        )
        fin_valid, fin_test_prediction, fin_iterations, fin_seconds = (
            fit_one_variant(
                X_financial,
                X_test_financial,
                labels,
                partition,
                seed,
            )
        )
        valid_indices = partition["outer_valid"]
        oof_raw[valid_indices] = raw_valid
        oof_financial[valid_indices] = fin_valid
        test_raw_by_fold.append(raw_test_prediction)
        test_financial_by_fold.append(fin_test_prediction)
        raw_auc = float(roc_auc_score(labels[valid_indices], raw_valid))
        fin_auc = float(roc_auc_score(labels[valid_indices], fin_valid))
        record = {
            "fold": fold_number,
            "outer_fit_rows": int(len(partition["outer_fit"])),
            "outer_valid_rows": int(len(valid_indices)),
            "inner_fit_rows": int(len(partition["inner_fit"])),
            "inner_stop_rows": int(len(partition["inner_stop"])),
            "outer_validation_used_for_early_stopping": False,
            "raw_selected_iterations": raw_iterations,
            "financial_selected_iterations": fin_iterations,
            "raw_auc": raw_auc,
            "financial_auc": fin_auc,
            "paired_auc_delta": fin_auc - raw_auc,
            "raw_fit_seconds": raw_seconds,
            "financial_fit_seconds": fin_seconds,
        }
        fold_training_records.append(record)
        print(
            f"Fold {fold_number}/5: raw={raw_auc:.8f}, "
            f"financial={fin_auc:.8f}, "
            f"delta={record['paired_auc_delta']:+.8f}, "
            f"iterations={raw_iterations}/{fin_iterations}, "
            f"seconds={raw_seconds:.1f}/{fin_seconds:.1f}",
            flush=True,
        )
finally:
    gpu_sampler.stop()

test_prediction_raw = np.mean(
    np.stack(test_raw_by_fold), axis=0, dtype=np.float64
).astype(np.float32)
test_prediction_financial = np.mean(
    np.stack(test_financial_by_fold), axis=0, dtype=np.float64
).astype(np.float32)
run_seconds = time.perf_counter() - run_start
run_completed_utc = datetime.now(timezone.utc).isoformat()
gpu_after = gpu_snapshot()
if np.isnan(oof_raw).any() or np.isnan(oof_financial).any():
    raise ValueError("OOF arrays contain uncovered rows")
for name, prediction in (
    ("raw OOF", oof_raw),
    ("financial OOF", oof_financial),
    ("raw test", test_prediction_raw),
    ("financial test", test_prediction_financial),
):
    if (
        not np.isfinite(prediction).all()
        or (prediction < 0).any()
        or (prediction > 1).any()
    ):
        raise ValueError(f"Invalid prediction values in {name}")

print(f"Total training runtime: {run_seconds:.1f} seconds")
print("GPU samples captured during training:", len(gpu_sampler.samples))


--- code cell 10 start ---
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Fold 1/5: raw=0.74979512, financial=0.75670954, delta=+0.00691442, iterations=613/699, seconds=56.3/79.5
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Fold 2/5: raw=0.75445683, financial=0.76087749, delta=+0.00642066, iterations=676/606, seconds=91.7/83.9
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented 

## 6. Recompute OOF metrics and verify artifacts

The advancement screen is predeclared. It requires positive pooled gain, at least +0.0002 mean paired fold gain, and improvement in at least four folds. A pass still requires independent-seed replication before acceptance. OOF AUC and official leaderboard scores remain separate.

In [6]:
fold_rows = []
for fold_number in range(1, N_SPLITS + 1):
    mask = fold_ids == fold_number
    raw_auc = float(roc_auc_score(labels[mask], oof_raw[mask]))
    financial_auc = float(roc_auc_score(labels[mask], oof_financial[mask]))
    fold_rows.append(
        {
            "fold": fold_number,
            "raw_auc": raw_auc,
            "financial_auc": financial_auc,
            "paired_auc_delta": financial_auc - raw_auc,
        }
    )
pooled_raw_auc = float(roc_auc_score(labels, oof_raw))
pooled_financial_auc = float(roc_auc_score(labels, oof_financial))
pooled_delta = pooled_financial_auc - pooled_raw_auc
mean_fold_delta = float(
    np.mean([row["paired_auc_delta"] for row in fold_rows])
)
positive_folds = int(
    sum(row["paired_auc_delta"] > 0 for row in fold_rows)
)
passes_screen = (
    pooled_delta > 0
    and mean_fold_delta >= 0.0002
    and positive_folds >= 4
)
decision = (
    "NEEDS_INDEPENDENT_SEED_REPLICATION" if passes_screen else "REJECTED"
)

independently_recomputed_fold_ids = np.full(
    len(labels), -1, dtype=np.int8
)
independent_cv = StratifiedKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=OUTER_SEED
)
for fold_number, (_, valid_indices) in enumerate(
    independent_cv.split(np.zeros(len(labels)), labels), start=1
):
    independently_recomputed_fold_ids[valid_indices] = fold_number
if not np.array_equal(fold_ids, independently_recomputed_fold_ids):
    raise ValueError("Outer folds differ from independent recomputation")
if not np.array_equal(
    sample_submission[ID_COLUMN].to_numpy(),
    test_raw[ID_COLUMN].to_numpy(),
):
    raise ValueError("Test/sample ID order changed during training")
if len(test_raw) != len(test_prediction_raw) or len(test_raw) != len(
    test_prediction_financial
):
    raise ValueError("Test prediction row count mismatch")
if any(
    row["outer_validation_used_for_early_stopping"]
    for row in fold_training_records
):
    raise ValueError("Outer validation influenced early stopping")

results = {
    "experiment": "EXP02_FINANCIAL_BEHAVIOUR",
    "run_id": RUN_ID,
    "decision": decision,
    "advancement_screen": {
        "positive_pooled_auc_delta": pooled_delta > 0,
        "minimum_mean_fold_delta": 0.0002,
        "observed_mean_fold_delta": mean_fold_delta,
        "required_positive_folds": 4,
        "observed_positive_folds": positive_folds,
        "passed": passes_screen,
        "independent_seed_replication_complete": False,
    },
    "train_rows": int(len(labels)),
    "test_rows": int(len(test_raw)),
    "predictor_columns_including_id": int(len(predictor_columns)),
    "outer_folds": N_SPLITS,
    "outer_seed": OUTER_SEED,
    "pooled_oof_auc": {
        "raw_baseline": pooled_raw_auc,
        "financial_features": pooled_financial_auc,
    },
    "pooled_oof_auc_delta": pooled_delta,
    "mean_paired_fold_auc_delta": mean_fold_delta,
    "positive_folds": positive_folds,
    "fold_metrics": fold_rows,
    "official_leaderboard_score": "NOT_EVALUATED",
    "submission_created": False,
    "runtime_seconds": run_seconds,
    "started_utc": run_started_utc,
    "completed_utc": run_completed_utc,
    "checks": {
        "input_hashes_match_course_package": True,
        "sample_test_id_alignment": True,
        "train_test_ids_disjoint": True,
        "fold_assignments_recomputed": True,
        "inner_splits_inside_outer_training": True,
        "outer_validation_not_used_for_early_stopping": True,
        "feature_formulas_use_only_supplied_columns": True,
        "target_and_id_excluded_from_features": True,
        "prediction_arrays_finite_in_unit_interval": True,
        "gpu_required_and_detected": True,
        "in_training_gpu_telemetry_captured": True,
    },
}
feature_manifest = {
    "experiment": "EXP02_FINANCIAL_BEHAVIOUR",
    "data_source": "provided course train.csv and test.csv only",
    "features": [
        {
            "name": name,
            "formula": formula,
            "source_columns": [
                part.strip() for part in formula.split("/")
            ],
            "uses_target": False,
            "uses_test_rows_to_fit": False,
            "fit_policy": "deterministic rowwise formula; no learned state",
        }
        for name, formula in FINANCIAL_FORMULAS.items()
    ],
    "excluded": [
        "SK_ID_CURR",
        "TARGET",
        "outside datasets",
        "fee/product constants",
        "pooled train/test statistics",
        "manual prediction rules",
    ],
    "invalid_denominator_policy": "non-finite or denominator <= 0 yields NaN",
}

np.save(RUN_DIR / "fold_ids.npy", fold_ids, allow_pickle=False)
np.save(RUN_DIR / "oof_raw_baseline.npy", oof_raw, allow_pickle=False)
np.save(RUN_DIR / "oof_financial_features.npy", oof_financial, allow_pickle=False)
np.save(
    RUN_DIR / "test_ids.npy",
    test_raw[ID_COLUMN].to_numpy(),
    allow_pickle=False,
)
np.save(
    RUN_DIR / "test_raw_baseline.npy",
    test_prediction_raw,
    allow_pickle=False,
)
np.save(
    RUN_DIR / "test_financial_features.npy",
    test_prediction_financial,
    allow_pickle=False,
)
(RUN_DIR / "results.json").write_text(
    json.dumps(results, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
pd.DataFrame(fold_rows).to_csv(
    RUN_DIR / "fold_metrics.csv", index=False, float_format="%.12g"
)
(RUN_DIR / "feature_manifest.json").write_text(
    json.dumps(feature_manifest, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
print(json.dumps(results, indent=2, allow_nan=False))


--- code cell 12 start ---
{
  "experiment": "EXP02_FINANCIAL_BEHAVIOUR",
  "run_id": "financial_ratios_seed20261105",
  "decision": "NEEDS_INDEPENDENT_SEED_REPLICATION",
  "advancement_screen": {
    "positive_pooled_auc_delta": true,
    "minimum_mean_fold_delta": 0.0002,
    "observed_mean_fold_delta": 0.006296821541723063,
    "required_positive_folds": 4,
    "observed_positive_folds": 5,
    "passed": true,
    "independent_seed_replication_complete": false
  },
  "train_rows": 171202,
  "test_rows": 61500,
  "predictor_columns_including_id": 33,
  "outer_folds": 5,
  "outer_seed": 20261105,
  "pooled_oof_auc": {
    "raw_baseline": 0.7517077237843034,
    "financial_features": 0.7580438653796194
  },
  "pooled_oof_auc_delta": 0.006336141595316036,
  "mean_paired_fold_auc_delta": 0.006296821541723063,
  "positive_folds": 5,
  "fold_metrics": [
    {
      "fold": 1,
      "raw_auc": 0.7497951206603107,
      "financial_auc": 0.7567095368878671,
      "paired_auc_delta": 0.006914

## 7. Write the report and hash manifest

The final cell reloads saved predictions and recomputes fold/metric checks before writing the report and manifest. It creates no CSV submission.

In [7]:
def notebook_source_sha256(path: Path) -> str:
    document = json.loads(path.read_text(encoding="utf-8"))
    source_cells = [
        {
            "cell_type": cell.get("cell_type"),
            "source": "".join(cell.get("source", [])),
        }
        for cell in document.get("cells", [])
    ]
    payload = json.dumps(
        source_cells,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

notebook_hash = notebook_source_sha256(NOTEBOOK_PATH)
run_manifest = {
    "experiment": "EXP02_FINANCIAL_BEHAVIOUR",
    "run_id": RUN_ID,
    "raw_input_sha256": input_sha256,
    "notebook_source_sha256": notebook_hash,
    "outer_folds": N_SPLITS,
    "outer_seed": OUTER_SEED,
    "inner_stopping_fraction": INNER_STOP_FRACTION,
    "inner_seed_rule": "20261102 + 1-based outer fold number",
    "model_seed_rule": "20261103 + 1-based fold; same for paired variants",
    "model": {
        **MODEL_PARAMS,
        "max_iterations": MAX_ITERATIONS,
        "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    },
    "libraries": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "catboost": catboost.__version__,
    },
    "gpu_before": gpu_before,
    "gpu_during_samples": gpu_sampler.samples,
    "gpu_during_max_utilization_pct": max(
        int(row["utilization_gpu_pct"]) for row in gpu_sampler.samples
    ),
    "gpu_during_max_memory_used_mib": max(
        int(row["memory_used_mib"]) for row in gpu_sampler.samples
    ),
    "gpu_after": gpu_after,
    "fold_training_records": fold_training_records,
    "external_data_or_prediction_cache_used": False,
    "manual_predictions_used": False,
    "submission_created": False,
}
(RUN_DIR / "run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

if abs(float(roc_auc_score(labels, oof_raw)) - pooled_raw_auc) > 1e-12:
    raise ValueError("Baseline pooled OOF AUC failed recomputation")
if (
    abs(float(roc_auc_score(labels, oof_financial)) - pooled_financial_auc)
    > 1e-12
):
    raise ValueError("Financial pooled OOF AUC failed recomputation")
saved_folds = np.load(RUN_DIR / "fold_ids.npy", allow_pickle=False)
if not np.array_equal(saved_folds, independently_recomputed_fold_ids):
    raise ValueError("Saved fold IDs failed reload verification")
for filename, expected in (
    ("oof_raw_baseline.npy", oof_raw),
    ("oof_financial_features.npy", oof_financial),
    ("test_raw_baseline.npy", test_prediction_raw),
    ("test_financial_features.npy", test_prediction_financial),
):
    loaded = np.load(RUN_DIR / filename, allow_pickle=False)
    if not np.array_equal(loaded, expected):
        raise ValueError(f"Saved prediction array failed reload verification: {filename}")

run_manifest_path = RUN_DIR / "run_manifest.json"
if WRITE_PRIMARY_SUMMARY:
    (EXPERIMENT_DIR / "RESULTS.json").write_text(
        json.dumps(results, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    pd.DataFrame(fold_rows).to_csv(
        EXPERIMENT_DIR / "FOLD_METRICS.csv",
        index=False,
        float_format="%.12g",
    )
    (EXPERIMENT_DIR / "FEATURE_MANIFEST.json").write_text(
        json.dumps(feature_manifest, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    fold_table = "\n".join(
        f"| {row['fold']} | {row['raw_auc']:.8f} | "
        f"{row['financial_auc']:.8f} | {row['paired_auc_delta']:+.8f} |"
        for row in fold_rows
    )
    report_lines = [
        "# EXP02 - Financial behaviour and affordability",
        "",
        "## Decision",
        "",
        f"**{decision}**. The independent-seed replication is not complete, "
        "so this is not an accepted model or submission candidate. No "
        "submission was created. Official leaderboard score: **NOT EVALUATED**.",
        "",
        "## Hypothesis and rule compliance",
        "",
        "A predeclared financial-ratio block was compared with the same "
        "raw-feature CatBoost model. Inputs matched the course package "
        "hashes. No external data, undocumented fee/product constants, "
        "validation/test covariate fitting, target encodings, manual "
        "prediction overrides, or public-score tuning were used.",
        "",
        "## Results",
        "",
        "| Metric | Raw-feature baseline | Financial feature model | Delta |",
        "|---|---:|---:|---:|",
        f"| Pooled 5-fold OOF AUC | {pooled_raw_auc:.8f} | "
        f"{pooled_financial_auc:.8f} | {pooled_delta:+.8f} |",
        f"| Mean paired fold AUC | - | - | {mean_fold_delta:+.8f} |",
        "",
        "| Fold | Raw AUC | Financial AUC | Paired delta |",
        "|---:|---:|---:|---:|",
        fold_table,
        "",
        f"Screen: positive pooled delta = {pooled_delta > 0}; mean "
        f"paired delta >= 0.0002 = {mean_fold_delta >= 0.0002}; "
        f"positive folds = {positive_folds}/5. A pass still requires "
        "independent-seed replication.",
        "",
        "## Protocol and leakage checks",
        "",
        "Five stratified outer folds (seed 42) share inner 10% stopping "
        "splits. Iteration selection uses only the inner split inside "
        "each outer training fold; final models refit on the full outer "
        "training fold. Validation labels are not used in early stopping. "
        "Feature formulas and checks are in `FEATURE_MANIFEST.json`.",
        "",
        "## Hardware and integrity",
        "",
        f"GPU: `{gpu_before[0]['name']}`. During-training polling: "
        f"{len(gpu_sampler.samples)} samples; max sampled utilization "
        f"{run_manifest['gpu_during_max_utilization_pct']}%; max sampled "
        f"VRAM {run_manifest['gpu_during_max_memory_used_mib']} MiB of "
        f"{gpu0_before['memory_total_mib']} MiB. These are sampled "
        f"values, not guaranteed true peaks. Total run time: "
        f"{run_seconds:.1f} seconds.",
        "",
        "Input, notebook source, and artifact SHA-256 values are in "
        "`SHA256_MANIFEST.json`. The notebook independently recomputed "
        "folds, OOF metrics, saved arrays, and test/sample ID alignment. "
        "It generated test prediction arrays only; no submission CSV.",
        "",
        "## Next step",
        "",
        "If the predeclared screen passes, run the frozen pipeline with "
        "a new outer seed as an independent replication. Otherwise reject "
        "this feature block. Do not select or edit rows based on leaderboard feedback.",
    ]
    (EXPERIMENT_DIR / "EXP02_REPORT.md").write_text(
        "\n".join(report_lines) + "\n", encoding="utf-8"
    )
    root_hashes = {
        name: sha256_file(EXPERIMENT_DIR / name)
        for name in (
            "EXPERIMENT_PROTOCOL.md",
            "RESULTS.json",
            "FOLD_METRICS.csv",
            "FEATURE_MANIFEST.json",
            "EXP02_REPORT.md",
        )
    }
    run_hashes = {
        path.name: sha256_file(path)
        for path in sorted(RUN_DIR.iterdir())
        if path.is_file()
    }
    manifest = {
        "version": 1,
        "experiment": "EXP02_FINANCIAL_BEHAVIOUR",
        "run_id": RUN_ID,
        "notebook_source_sha256": notebook_hash,
        "raw_input_sha256": input_sha256,
        "root_artifact_sha256": root_hashes,
        "run_artifact_sha256": run_hashes,
        "note": "Run arrays are local research artifacts ignored by Git; no submission CSV was created.",
    }
    (EXPERIMENT_DIR / "SHA256_MANIFEST.json").write_text(
        json.dumps(manifest, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    print("Decision:", decision)
    print("Results report:", EXPERIMENT_DIR / "EXP02_REPORT.md")
    print("Hash manifest:", EXPERIMENT_DIR / "SHA256_MANIFEST.json")
    print(
        "Root/run/input artifacts hashed:",
        len(root_hashes) + len(run_hashes) + len(input_sha256),
    )
else:
    run_hashes = {
        path.name: sha256_file(path)
        for path in sorted(RUN_DIR.iterdir())
        if path.is_file()
    }
    run_manifest["run_artifact_sha256"] = run_hashes
    run_manifest_path.write_text(
        json.dumps(run_manifest, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    (RUN_DIR / "SHA256_MANIFEST.json").write_text(
        json.dumps(
            {
                "run_id": RUN_ID,
                "notebook_source_sha256": notebook_hash,
                "raw_input_sha256": input_sha256,
                "run_artifact_sha256": run_hashes,
            },
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )
    print("Secondary run complete; primary experiment summaries were preserved.")


--- code cell 14 start ---
Decision: NEEDS_INDEPENDENT_SEED_REPLICATION
Results report: D:\gci\truth\experiments\exp02_financial_behaviour_replication_seed20261105\EXP02_REPORT.md
Hash manifest: D:\gci\truth\experiments\exp02_financial_behaviour_replication_seed20261105\SHA256_MANIFEST.json
Root/run/input artifacts hashed: 18
